# 🧱 Python Classes & Objects from Scratch

You've been *using* objects since your first line of Python: a string, a list and a DataFrame are all objects. This notebook teaches you to **build your own**, starting from an empty class and ending with small real-world classes that use inheritance, properties, dataclasses and pandas.

### How this notebook works

Every topic follows the same loop, so you get lots of repetition:

1. **📖 Learn**: a short explanation.
2. **👀 See it**: worked examples you run and read. Change them and re-run.
3. **✏️ Your turn**: several exercises. Each has a stub (`pass` or a comment) for you to replace, and a **checker** underneath that prints ✅ or ❌ and says *which* behaviour failed.

> **Rules of the road**
> * Run the **setup cell** first, then go top to bottom. Classes you define stay in memory, so if you rename something, re-run the cell that defines it.
> * Each exercise is self-contained. Where it needs another class, that class is already written for you in the same cell.
> * If you see `AttributeError` or `TypeError` in a ❌ message, read it. It's usually telling you exactly what's missing.
> * Stuck? Re-read the "See it" cells, then check the **answer key** at the bottom. Try first.
> * Exercises marked ⭐ are bonus.

### Roadmap

| Part | Topic |
|---|---|
| 1 | Objects you already use: attributes & methods |
| 2 | Your first class: blueprint vs object |
| 3 | `__init__` and `self` |
| 4 | Methods |
| 5 | Default values, validation & the mutable-default trap |
| 6 | Class attributes vs instance attributes |
| 7 | Special ("dunder") methods: `__str__`, `__eq__`, `__add__`, ... |
| 8 | Encapsulation & `@property` |
| 9 | `@classmethod` & `@staticmethod` |
| 10 | Inheritance, `super()` & polymorphism |
| 11 | Composition: objects that contain objects |
| 12 | Container protocols & `@dataclass` |
| 13 | Mini-projects |
| 14 | Classes + pandas |

## Setup (run this first)

This defines `run_checks`, the little test-runner used by every exercise. It takes pairs of *(description, test)*. A test is a `lambda` that should return `True`. If it returns something falsy or raises an error, you'll see which one failed. You don't need to understand the internals, but feel free to read it.

In [ ]:
import io
import contextlib

def run_checks(*tests):
    """Each test is (description, zero-argument function returning True)."""
    problems = []
    for desc, fn in tests:
        try:
            ok = fn()
        except Exception as e:
            problems.append(f"   ✗ {desc}\n       -> {type(e).__name__}: {e}")
            continue
        if not ok:
            problems.append(f"   ✗ {desc}")
    if not problems:
        print(f"✅ Nice! All {len(tests)} checks passed.")
    else:
        print(f"❌ {len(problems)} of {len(tests)} checks failed:")
        for p in problems:
            print(p)

def raises(exc_type, fn):
    """True if calling fn() raises exc_type (and nothing else)."""
    try:
        fn()
    except exc_type:
        return True
    except Exception:
        return False
    return False

def printed(fn):
    """Return everything fn() prints, as a string."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        fn()
    return buf.getvalue()

print("Setup complete ✔")

---
# Part 1 · Objects you already use

## 📖 Learn

In Python, **everything is an object**: a bundle of

* **data**, called *attributes* (`z.real`: no parentheses), and
* **behaviour**, called *methods* (`s.upper()`: parentheses, because it's a function attached to the object).

You call both with the **dot**: `object.thing`. Every object has a **type** (the *class* it was made from). `"hi"` is an object of class `str`; `[1, 2]` is an object of class `list`. Writing your own classes just means defining new types.

## 👀 See it

In [ ]:
s = "hello world"
print(type(s))              # the class this object was made from
print(s.upper())            # a METHOD: behaviour attached to the object
print(s.split())            # another method; it returns a new list

nums = [3, 1, 2]
nums.append(4)              # a method that CHANGES the object
print(nums, len(nums))

z = 3 + 4j
print(z.real, z.imag)       # ATTRIBUTES: plain data stored on the object
print(z.conjugate())        # ...and a method

print(isinstance(s, str), isinstance(nums, list), isinstance(s, list))
print([name for name in dir(s) if not name.startswith("_")][:8])   # what can a str do?

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 1.1 — Use a string METHOD to turn `text` into upper case
text = "the quick brown fox"
shout = None   # <- your code here
run_checks(("shout == 'THE QUICK BROWN FOX'", lambda: shout == "THE QUICK BROWN FOX"))

In [ ]:
# ✏️ Exercise 1.2 — Use a string METHOD to split `text` into a list of words
text = "the quick brown fox"
words = None   # <- your code here
run_checks(("words == ['the', 'quick', 'brown', 'fox']", lambda: words == ["the", "quick", "brown", "fox"]))

In [ ]:
# ✏️ Exercise 1.3 — Call a list METHOD that sorts `nums` IN PLACE (don't create a new list)
nums = [5, 3, 8, 1]
pass   # <- your code here
run_checks(("nums is now [1, 3, 5, 8]", lambda: nums == [1, 3, 5, 8]))

In [ ]:
# ✏️ Exercise 1.4 — Get the TYPE of the value 3.5
kind = None   # <- your code here
run_checks(("kind is float", lambda: kind is float))

In [ ]:
# ✏️ Exercise 1.5 — Attribute vs method on a complex number
z = 3 + 4j
imag_part = None   # <- the ATTRIBUTE holding the imaginary part. Hint: no parentheses.
run_checks(("imag_part == 4.0", lambda: imag_part == 4.0))

---
# Part 2 · Your first class

## 📖 Learn

* A **class** is a *blueprint*: it describes what a kind of thing is and does.
* An **object** (or **instance**) is one particular thing built from that blueprint.

```python
class Dog:        # class names use CamelCase
    pass          # "pass" = an empty body, for now

rex = Dog()       # calling the class creates a new object
```

Each object keeps its **own** attributes, which you can attach with the dot: `rex.name = "Rex"`. Two objects built from the same class are different objects, and `is` checks whether two names point at the *same* one.

## 👀 See it

In [ ]:
class Dog:
    pass

rex = Dog()                     # create ("instantiate") an object
fido = Dog()
rex.name = "Rex"                # attributes belong to each object separately
rex.age = 3
fido.name = "Fido"
print(rex.name, rex.age, fido.name)
print(type(rex), isinstance(rex, Dog), rex is fido)

In [ ]:
# Two NAMES can point at ONE object (aliasing):
a = Dog()
b = a              # no new object. b is just another name for the same one
b.name = "Shared"
print(a.name)      # changed through b!
print(a is b)

# ...whereas two separate Dog() calls give two separate objects
c, d = Dog(), Dog()
print(c is d, c == d)    # by default == also means "same object"

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 2.1 — Define an empty class called Cat
# write your class here
run_checks(
    ("Cat() makes a Cat object", lambda: isinstance(Cat(), Cat)),
    ("two Cat() calls make different objects", lambda: Cat() is not Cat()),
    ("the class is named Cat", lambda: type(Cat()).__name__ == "Cat"),
)

In [ ]:
# ✏️ Exercise 2.2 — Attach attributes to two separate objects
class Book:
    pass

b1 = Book()
b2 = Book()
pass   # <- set b1.title = "Dune" and b2.title = "Emma"
run_checks(
    ("b1.title == 'Dune'", lambda: b1.title == "Dune"),
    ("b2.title == 'Emma'", lambda: b2.title == "Emma"),
    ("objects keep their own attributes", lambda: b1.title != b2.title),
)

In [ ]:
# ✏️ Exercise 2.3 — Create ONE Book object called my_book with three attributes:
# title (a string), author (a string) and pages (an int). Pick any values.
class Book:
    pass

my_book = None
run_checks(
    ("my_book is a Book", lambda: isinstance(my_book, Book)),
    ("title is a str", lambda: isinstance(my_book.title, str)),
    ("author is a str", lambda: isinstance(my_book.author, str)),
    ("pages is an int", lambda: isinstance(my_book.pages, int)),
)

In [ ]:
# ✏️ Exercise 2.4 — Predict, then run. Two names, ONE object:
class Book:
    pass

a = Book()
b = a
b.title = "Dune"
a.title = "Emma"
answer = None   # <- what is b.title now? Write the value you predict (a string)
run_checks(("your prediction matches what Python gives", lambda: answer == b.title))

In [ ]:
# ✏️ Exercise 2.5 — Predict, then run. By default, are two separate objects "equal"?
class Book:
    pass

answer = None   # <- write True or False for the value of  Book() == Book()
run_checks(("your prediction matches what Python gives", lambda: answer is (Book() == Book())))

---
# Part 3 · `__init__` and `self`

## 📖 Learn

Attaching attributes by hand (Part 2) is tedious and easy to forget. The special method **`__init__`** runs *automatically* every time an object is created, so it can set the object up:

```python
class Dog:
    def __init__(self, name, age):
        self.name = name        # store the argument on the object
        self.age = age

rex = Dog("Rex", 3)             # __init__ runs here
```

* **`self`** is the object being built (or later, the object you called a method on). Python passes it automatically. You never pass `self` yourself, but you *always* write it as the first parameter.
* `self.name = name` means "give *this object* an attribute called `name`". The left `name` is the attribute; the right `name` is the parameter.
* Without the `self.`, `name` would be a throw-away local variable that disappears when `__init__` ends.

## 👀 See it

In [ ]:
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

rex = Dog("Rex", 3)
fido = Dog("Fido", 5)
print(rex.name, rex.age)
print(fido.name, fido.age)

try:
    Dog("Lonely")        # too few arguments: Python tells you what's missing
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# __init__ can also compute things and start attributes at fixed values
class Playlist:
    def __init__(self, name):
        self.name = name
        self.songs = []                 # every new playlist starts empty
        self.title = name.upper()       # ...or derive a value from the arguments

p = Playlist("road trip")
print(p.name, p.songs, p.title)

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 3.1 — A Point with x and y
class Point:
    pass   # <- write __init__(self, x, y) and store both values

run_checks(
    ("Point(3, 4).x == 3", lambda: Point(3, 4).x == 3),
    ("Point(3, 4).y == 4", lambda: Point(3, 4).y == 4),
    ("each Point keeps its own values", lambda: (Point(1, 2).x, Point(5, 6).x) == (1, 5)),
)

In [ ]:
# ✏️ Exercise 3.2 — A Book with title, author and pages
class Book:
    pass   # <- write __init__(self, title, author, pages)

run_checks(
    ("title stored", lambda: Book("Dune", "Herbert", 412).title == "Dune"),
    ("author stored", lambda: Book("Dune", "Herbert", 412).author == "Herbert"),
    ("pages stored", lambda: Book("Dune", "Herbert", 412).pages == 412),
)

In [ ]:
# ✏️ Exercise 3.3 — A Rectangle with width and height
class Rectangle:
    pass   # <- write __init__(self, width, height)

run_checks(
    ("width stored", lambda: Rectangle(3, 4).width == 3),
    ("height stored", lambda: Rectangle(3, 4).height == 4),
)

In [ ]:
# ✏️ Exercise 3.4 — Compute an attribute inside __init__
# Person(first, last) should store first, last AND full_name ("Ada Lovelace")
class Person:
    pass   # <- write __init__(self, first, last)

run_checks(
    ("first stored", lambda: Person("Ada", "Lovelace").first == "Ada"),
    ("last stored", lambda: Person("Ada", "Lovelace").last == "Lovelace"),
    ("full_name computed", lambda: Person("Ada", "Lovelace").full_name == "Ada Lovelace"),
)

In [ ]:
# ✏️ Exercise 3.5 — Attributes that start at a fixed value
# Counter() takes NO arguments. It starts with count = 0.
class Counter:
    pass   # <- write __init__(self)

run_checks(
    ("Counter() works with no arguments", lambda: Counter() is not None),
    ("count starts at 0", lambda: Counter().count == 0),
)

In [ ]:
# ✏️ Exercise 3.6 — Every object needs its OWN list
# Playlist(name) stores name and starts with an empty list called songs.
# (Two playlists must NOT share one list.)
class Playlist:
    pass   # <- write __init__(self, name)

def _separate_lists():
    a, b = Playlist("A"), Playlist("B")
    a.songs.append("x")
    return b.songs == []

run_checks(
    ("name stored", lambda: Playlist("Road trip").name == "Road trip"),
    ("songs starts as an empty list", lambda: Playlist("Road trip").songs == []),
    ("playlists don't share their list", _separate_lists),
)

---
# Part 4 · Methods

## 📖 Learn

A **method** is a function defined *inside* a class. Its first parameter is always `self`, so it can read and change the object's attributes:

```python
class Dog:
    def __init__(self, name):
        self.name = name

    def bark(self):                  # self is passed automatically
        return f"{self.name} says Woof!"

Dog("Rex").bark()                    # -> "Rex says Woof!"
```

Two kinds of method you'll write constantly:

* **Query** methods *return* something computed from the attributes (`area()`).
* **Command** methods *change* the object's attributes (`increment()`), often returning nothing.

A method can call other methods on the same object with `self.other_method()`.

## 👀 See it

In [ ]:
class Account:
    def __init__(self, owner, balance):
        self.owner = owner
        self.balance = balance

    def deposit(self, amount):          # a COMMAND: changes state
        self.balance += amount

    def summary(self):                  # a QUERY: returns a value
        return f"{self.owner}: ${self.balance}"

    def deposit_and_report(self, amount):
        self.deposit(amount)            # call another method on self
        return self.summary()

acct = Account("Ana", 100)
acct.deposit(50)
print(acct.summary())
print(acct.deposit_and_report(25))
print(acct.deposit(5))                  # commands return None

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 4.1 — greet() returns  "Hello, <name>!"
class Greeter:
    def __init__(self, name):
        self.name = name

    pass   # <- write greet(self)

run_checks(
    ("greet() for Ana", lambda: Greeter("Ana").greet() == "Hello, Ana!"),
    ("greet() for Bo", lambda: Greeter("Bo").greet() == "Hello, Bo!"),
)

In [ ]:
# ✏️ Exercise 4.2 — area() and perimeter()
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    pass   # <- write area(self) and perimeter(self)

run_checks(
    ("area of 3x4 is 12", lambda: Rectangle(3, 4).area() == 12),
    ("perimeter of 3x4 is 14", lambda: Rectangle(3, 4).perimeter() == 14),
    ("area of 5x5 is 25", lambda: Rectangle(5, 5).area() == 25),
)

In [ ]:
# ✏️ Exercise 4.3 — Command methods that CHANGE state
# increment() adds 1 to count; reset() sets it back to 0. Neither returns anything.
class Counter:
    def __init__(self):
        self.count = 0

    pass   # <- write increment(self) and reset(self)

def _two_incs():
    c = Counter(); c.increment(); c.increment(); return c.count == 2

def _reset():
    c = Counter(); c.increment(); c.reset(); return c.count == 0

run_checks(
    ("increment() twice -> count == 2", _two_incs),
    ("reset() -> count == 0", _reset),
    ("increment() returns None", lambda: Counter().increment() is None),
)

In [ ]:
# ✏️ Exercise 4.4 — Use math.pi inside methods
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius

    pass   # <- write area(self) and circumference(self)

run_checks(
    ("area of radius 1", lambda: abs(Circle(1).area() - math.pi) < 1e-9),
    ("area of radius 2", lambda: abs(Circle(2).area() - 4 * math.pi) < 1e-9),
    ("circumference of radius 1", lambda: abs(Circle(1).circumference() - 2 * math.pi) < 1e-9),
)

In [ ]:
# ✏️ Exercise 4.5 — A method with extra parameters, plus a method that uses a list
# add_grade(g) appends g to grades.  average() returns the mean (or 0 if there are no grades).
class Student:
    def __init__(self, name):
        self.name = name
        self.grades = []

    pass   # <- write add_grade(self, grade) and average(self)

def _avg():
    s = Student("Ana"); s.add_grade(80); s.add_grade(90); return s.average() == 85

run_checks(
    ("average of 80 and 90 is 85", _avg),
    ("average with no grades is 0", lambda: Student("Bo").average() == 0),
    ("add_grade stores the grade", lambda: (lambda s: (s.add_grade(70), s.grades)[1])(Student("C")) == [70]),
)

In [ ]:
# ✏️ Exercise 4.6 — A method that calls another method
# describe() returns e.g. "3x4 rectangle with area 12", using self.area()
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    def area(self):
        return self.width * self.height

    pass   # <- write describe(self)

run_checks(
    ("describe 3x4", lambda: Rectangle(3, 4).describe() == "3x4 rectangle with area 12"),
    ("describe 5x2", lambda: Rectangle(5, 2).describe() == "5x2 rectangle with area 10"),
)

---
# Part 5 · Default values, validation & the mutable-default trap

## 📖 Learn

* **Defaults:** `def __init__(self, name, price=0.0)` makes `price` optional. Callers can also use keywords: `Item("pen", quantity=5)`.
* **Validation:** `__init__` is the best place to refuse nonsense. Raise an error with `raise ValueError("...")` so bad objects are never created.
* **⚠️ The mutable-default trap:** a default like `items=[]` is created **once**, when the function is defined, and then *shared by every call*. Use `None` as the default and make the list inside:

```python
def __init__(self, items=None):
    self.items = items if items is not None else []
```

## 👀 See it

In [ ]:
class Item:
    def __init__(self, name, price=0.0, quantity=1):
        if price < 0:
            raise ValueError("price cannot be negative")
        self.name, self.price, self.quantity = name, price, quantity

print(vars(Item("pen")))                       # vars() shows an object's attributes
print(vars(Item("pen", 2.5, quantity=10)))
try:
    Item("pen", -1)
except ValueError as e:
    print("ValueError:", e)

In [ ]:
# THE TRAP: the default list is shared!
class BadBasket:
    def __init__(self, items=[]):
        self.items = items

a, b = BadBasket(), BadBasket()
a.items.append("apple")
print(b.items)                 # b got an apple too!

# THE FIX:
class GoodBasket:
    def __init__(self, items=None):
        self.items = items if items is not None else []

a, b = GoodBasket(), GoodBasket()
a.items.append("apple")
print(b.items)

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 5.1 — Defaults: price defaults to 0.0, quantity defaults to 1
class Item:
    pass   # <- write __init__(self, name, price=..., quantity=...)

run_checks(
    ("Item('pen') uses both defaults", lambda: (Item("pen").price, Item("pen").quantity) == (0.0, 1)),
    ("Item('pen', 2.5, 3) sets both", lambda: (Item("pen", 2.5, 3).price, Item("pen", 2.5, 3).quantity) == (2.5, 3)),
    ("keyword argument works", lambda: Item("pen", quantity=5).quantity == 5),
    ("name stored", lambda: Item("pen").name == "pen"),
)

In [ ]:
# ✏️ Exercise 5.2 — Validation: a balance can't start negative
# Account(owner, balance=0). If balance < 0 raise ValueError.
class Account:
    pass   # <- write __init__(self, owner, balance=0)

run_checks(
    ("default balance is 0", lambda: Account("Ana").balance == 0),
    ("balance stored", lambda: Account("Ana", 50).balance == 50),
    ("owner stored", lambda: Account("Ana").owner == "Ana"),
    ("negative balance raises ValueError", lambda: raises(ValueError, lambda: Account("Ana", -5))),
)

In [ ]:
# ✏️ Exercise 5.3 — Validation with a range
# Percentage(value): value must be between 0 and 100 inclusive, otherwise ValueError.
class Percentage:
    pass   # <- write __init__(self, value)

run_checks(
    ("50 is fine", lambda: Percentage(50).value == 50),
    ("0 and 100 are fine", lambda: (Percentage(0).value, Percentage(100).value) == (0, 100)),
    ("-1 raises ValueError", lambda: raises(ValueError, lambda: Percentage(-1))),
    ("101 raises ValueError", lambda: raises(ValueError, lambda: Percentage(101))),
)

In [ ]:
# ✏️ Exercise 5.4 — FIX THE BUG: this Basket shares one list between all baskets.
# Rewrite __init__ so each basket gets its own list (use None as the default).
class Basket:
    def __init__(self, items=[]):      # <- buggy: fix me
        self.items = items

    def add(self, item):
        self.items.append(item)

def _independent():
    a, b = Basket(), Basket()
    a.add("apple")
    return b.items == []

run_checks(
    ("baskets don't share items", _independent),
    ("you can still pass a starting list", lambda: Basket(["pear"]).items == ["pear"]),
    ("add works", lambda: (lambda b: (b.add("x"), b.items)[1])(Basket()) == ["x"]),
)

---
# Part 6 · Class attributes vs instance attributes

## 📖 Learn

* An **instance attribute** is set with `self.x = ...` and belongs to *one object*.
* A **class attribute** is written directly in the class body and is *shared* by all objects (and reachable from the class itself: `Dog.species`).

```python
class Dog:
    species = "Canis familiaris"      # class attribute: shared
    count = 0

    def __init__(self, name):
        self.name = name              # instance attribute: per object
        Dog.count += 1                # update the SHARED counter through the class name
```

Reading `obj.attr` looks on the object first, then on its class. Assigning `obj.attr = ...` always creates/changes the *object's* attribute (it **shadows** the class one without changing it).

## 👀 See it

In [ ]:
class Dog:
    species = "Canis familiaris"
    count = 0

    def __init__(self, name):
        self.name = name
        Dog.count += 1

a, b = Dog("Rex"), Dog("Fido")
print(Dog.species, a.species, b.species)   # one shared value
print(Dog.count)                           # 2: shared counter

a.species = "Wolf"                         # creates an INSTANCE attribute on a only
print(a.species, b.species, Dog.species)   # shadowing: class value unchanged

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 6.1 — Add a class attribute  species = "Canis familiaris"
class Dog:
    pass   # <- class attribute goes here, directly in the class body

    def __init__(self, name):
        self.name = name

run_checks(
    ("reachable from the class", lambda: Dog.species == "Canis familiaris"),
    ("reachable from an object", lambda: Dog("Rex").species == "Canis familiaris"),
    ("instance attribute still works", lambda: Dog("Rex").name == "Rex"),
)

In [ ]:
# ✏️ Exercise 6.2 — Count how many Players have been created
# Give the class an attribute  count = 0  and increase it inside __init__.
class Player:
    pass   # <- write count = 0 and __init__(self, name)

run_checks(
    ("count starts at 0", lambda: Player.count == 0),
    ("count goes up as objects are made", lambda: (Player("a"), Player("b"), Player("c"), Player.count)[-1] == 3),
    ("name stored", lambda: Player("zed").name == "zed"),
)

In [ ]:
# ✏️ Exercise 6.3 — Predict, then run. Shadowing:
class A:
    x = 1

a = A()
a.x = 2
b = A()
answer = None   # <- a tuple (A.x, a.x, b.x). Write the three values you predict, e.g. (9, 9, 9)
run_checks(("your prediction matches", lambda: answer == (A.x, a.x, b.x)))

In [ ]:
# ✏️ Exercise 6.4 — A shared setting that an object can override
# raise_pct = 0.05 is a class attribute. apply_raise() multiplies salary by (1 + self.raise_pct).
class Employee:
    pass   # <- class attribute raise_pct, __init__(self, name, salary), apply_raise(self)

def _default():
    e = Employee("Ana", 1000); e.apply_raise(); return round(e.salary, 2) == 1050

def _override():
    e = Employee("Ana", 1000); e.raise_pct = 0.10; e.apply_raise(); return round(e.salary, 2) == 1100

run_checks(
    ("class attribute exists", lambda: Employee.raise_pct == 0.05),
    ("5% raise by default", _default),
    ("one object can override raise_pct", _override),
    ("overriding one object doesn't change the class", lambda: (lambda e: (setattr(e, "raise_pct", 0.5), Employee.raise_pct)[1])(Employee("x", 1)) == 0.05),
)

In [ ]:
# ✏️ Exercise 6.5 — Auto-numbering with a class attribute
# Every new Ticket gets id 1, 2, 3, ... Keep the next id in a class attribute  next_id = 1.
class Ticket:
    pass   # <- class attribute next_id, and __init__(self) that sets self.id and bumps next_id

def _ids():
    Ticket.next_id = 1
    t1, t2, t3 = Ticket(), Ticket(), Ticket()
    return (t1.id, t2.id, t3.id) == (1, 2, 3)

run_checks(
    ("next_id starts at 1", lambda: Ticket.next_id == 1),
    ("ids count up: 1, 2, 3", _ids),
    ("next_id is now 4", lambda: Ticket.next_id == 4),
)

---
# Part 7 · Special ("dunder") methods

## 📖 Learn

Methods named `__like_this__` ("double underscore", or **dunder**) are hooks Python calls for you when you use built-in syntax:

| You write | Python calls | Typical purpose |
|---|---|---|
| `print(obj)`, `str(obj)` | `obj.__str__()` | friendly text for users |
| `repr(obj)`, shown in lists / the REPL | `obj.__repr__()` | unambiguous text for developers |
| `a == b` | `a.__eq__(b)` | equality by *value* |
| `a < b` (and `sorted`) | `a.__lt__(b)` | ordering |
| `a + b` / `a * 3` | `a.__add__(b)` / `a.__mul__(3)` | arithmetic |
| `len(obj)` | `obj.__len__()` | size (also makes empty objects falsy) |

Tips: `__str__` and `__repr__` must **return** a string (not print it). If `__eq__` receives something of the wrong type, `return NotImplemented` (or `False`) instead of crashing.

## 👀 See it

In [ ]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Point({self.x}, {self.y})"

    def __eq__(self, other):
        if not isinstance(other, Point):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __add__(self, other):
        return Point(self.x + other.x, self.y + other.y)

    def __lt__(self, other):
        return (self.x, self.y) < (other.x, other.y)

p, q = Point(1, 2), Point(3, 4)
print(p)                                  # uses __repr__ (no __str__ defined)
print([p, q])
print(p == Point(1, 2), p == q, p == "nope")
print(p + q)
print(sorted([q, p]))

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = songs
    def __len__(self):
        return len(self.songs)

print(len(Playlist(["a", "b"])))
print(bool(Playlist([])), bool(Playlist(["a"])))   # an object with len 0 is falsy

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 7.1 — __str__
# str(Dog("Rex", 3)) should be  "Rex (age 3)"  (return the text; don't print it)
class Dog:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    pass   # <- write __str__(self)

run_checks(
    ("str(Dog('Rex', 3))", lambda: str(Dog("Rex", 3)) == "Rex (age 3)"),
    ("print() uses it", lambda: printed(lambda: print(Dog("Max", 5))) == "Max (age 5)\n"),
    ("f-strings use it", lambda: f"{Dog('Bo', 1)}" == "Bo (age 1)"),
)

In [ ]:
# ✏️ Exercise 7.2 — __repr__
# repr(Point(3, 4)) should be  "Point(3, 4)"
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    pass   # <- write __repr__(self)

run_checks(
    ("repr(Point(3, 4))", lambda: repr(Point(3, 4)) == "Point(3, 4)"),
    ("repr in a list", lambda: repr([Point(1, 2)]) == "[Point(1, 2)]"),
)

In [ ]:
# ✏️ Exercise 7.3 — __eq__ (equal by VALUE)
# Two Points are equal when x and y match. Comparing to a non-Point must give False.
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    pass   # <- write __eq__(self, other)

run_checks(
    ("same values are equal", lambda: Point(1, 2) == Point(1, 2)),
    ("different y not equal", lambda: not (Point(1, 2) == Point(1, 3))),
    ("!= works too", lambda: Point(1, 2) != Point(9, 9)),
    ("comparing with a tuple is False (no crash)", lambda: (Point(1, 2) == (1, 2)) is False),
)

In [ ]:
# ✏️ Exercise 7.4 — __add__
# Vector(1, 2) + Vector(3, 4) should be a NEW Vector(4, 6)
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return isinstance(other, Vector) and (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    pass   # <- write __add__(self, other)

def _no_mutation():
    a, b = Vector(1, 2), Vector(3, 4)
    a + b
    return a == Vector(1, 2) and b == Vector(3, 4)

run_checks(
    ("Vector(1, 2) + Vector(3, 4) == Vector(4, 6)", lambda: Vector(1, 2) + Vector(3, 4) == Vector(4, 6)),
    ("works with negatives", lambda: Vector(1, 1) + Vector(-1, -1) == Vector(0, 0)),
    ("returns a new object, doesn't change the originals", _no_mutation),
)

In [ ]:
# ✏️ Exercise 7.5 — __lt__ makes sorted() / min() / max() work
# Order Students by grade (lowest first).
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    pass   # <- write __lt__(self, other)

students = [Student("ann", 90), Student("bob", 72), Student("cy", 85)]
run_checks(
    ("sorted() orders by grade", lambda: [s.name for s in sorted(students)] == ["bob", "cy", "ann"]),
    ("min() is the lowest grade", lambda: min(students).name == "bob"),
    ("max() is the highest grade", lambda: max(students).name == "ann"),
)

In [ ]:
# ✏️ Exercise 7.6 — __len__ (and free truthiness!)
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    pass   # <- write __len__(self)

run_checks(
    ("len() counts songs", lambda: len(Playlist(["a", "b", "c"])) == 3),
    ("an empty playlist has len 0", lambda: len(Playlist([])) == 0),
    ("an empty playlist is falsy", lambda: not Playlist([])),
    ("a non-empty playlist is truthy", lambda: bool(Playlist(["a"])) is True),
)

In [ ]:
# ✏️ Exercise 7.7 ⭐ — __mul__ (scalar multiplication) and __neg__
# Vector(1, 2) * 3 == Vector(3, 6)  and  -Vector(1, -2) == Vector(-1, 2)
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __eq__(self, other):
        return isinstance(other, Vector) and (self.x, self.y) == (other.x, other.y)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

    pass   # <- write __mul__(self, k) and __neg__(self)

run_checks(
    ("Vector(1, 2) * 3", lambda: Vector(1, 2) * 3 == Vector(3, 6)),
    ("times 0", lambda: Vector(5, 5) * 0 == Vector(0, 0)),
    ("negation", lambda: -Vector(1, -2) == Vector(-1, 2)),
)

---
# Part 8 · Encapsulation & `@property`

## 📖 Learn

Python has no truly private attributes, but there's a strong **convention**: a name starting with an underscore (`self._balance`) means "internal: don't touch from outside".

When you want attribute-style access (`acct.balance`) *with* logic behind it, use **`@property`**:

```python
class Circle:
    def __init__(self, radius):
        self.radius = radius

    @property
    def area(self):                  # read like an attribute: c.area   (no parentheses!)
        return 3.14159 * self.radius ** 2
```

* A property with only a getter is **read-only**: assigning to it raises `AttributeError`.
* Add a **setter** with `@name.setter` to run validation whenever someone assigns. The value itself is stored in `self._name`.
* Because `__init__` also assigns through the property, validation happens at creation time too.

## 👀 See it

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius                  # goes through the setter below!

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if value < -273.15:
            raise ValueError("below absolute zero")
        self._celsius = value

    @property
    def kelvin(self):                           # computed on demand, read-only
        return self._celsius + 273.15

t = Temperature(25)
print(t.celsius, t.kelvin)
t.celsius = 30
print(t.kelvin)
for bad in (lambda: Temperature(-300), lambda: setattr(t, "kelvin", 1)):
    try:
        bad()
    except (ValueError, AttributeError) as e:
        print(type(e).__name__, "-", e)

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 8.1 — A read-only computed property
# Make `area` a @property (so you read r.area, not r.area()). It must be read-only.
class Rectangle:
    def __init__(self, width, height):
        self.width = width
        self.height = height

    pass   # <- write the property

def _tracks_changes():
    r = Rectangle(3, 4); r.width = 10; return r.area == 40

run_checks(
    ("r.area == 12", lambda: Rectangle(3, 4).area == 12),
    ("area follows later changes", _tracks_changes),
    ("assigning to area raises AttributeError", lambda: raises(AttributeError, lambda: setattr(Rectangle(3, 4), "area", 5))),
)

In [ ]:
# ✏️ Exercise 8.2 — A property computed from two attributes
# full_name should always be  "<first> <last>"  using the CURRENT first and last.
class Person:
    def __init__(self, first, last):
        self.first = first
        self.last = last

    pass   # <- write the property

def _updates():
    p = Person("Ada", "Lovelace"); p.first = "Grace"; return p.full_name == "Grace Lovelace"

run_checks(
    ("full_name", lambda: Person("Ada", "Lovelace").full_name == "Ada Lovelace"),
    ("updates when first changes", _updates),
)

In [ ]:
# ✏️ Exercise 8.3 — A property WITH a setter (validation)
# age must never be negative. Store the real value in self._age.
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age          # <- this line goes through YOUR setter

    pass   # <- write the @property getter and the @age.setter

def _set_ok():
    p = Person("Ana", 30); p.age = 31; return p.age == 31

run_checks(
    ("age readable", lambda: Person("Ana", 30).age == 30),
    ("assigning a new valid age works", _set_ok),
    ("negative age at creation raises ValueError", lambda: raises(ValueError, lambda: Person("Ana", -1))),
    ("negative age by assignment raises ValueError", lambda: raises(ValueError, lambda: setattr(Person("Ana", 30), "age", -5))),
    ("value is stored in _age", lambda: Person("Ana", 30)._age == 30),
)

In [ ]:
# ✏️ Exercise 8.4 — A property that converts units, with a setter
# celsius is a plain attribute. Add a `fahrenheit` property: F = C * 9/5 + 32.
# Setting fahrenheit should update celsius.
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    pass   # <- write the fahrenheit getter and setter

def _set_f():
    t = Temperature(0); t.fahrenheit = 212; return round(t.celsius, 6) == 100

run_checks(
    ("100 C is 212 F", lambda: Temperature(100).fahrenheit == 212),
    ("0 C is 32 F", lambda: Temperature(0).fahrenheit == 32),
    ("setting fahrenheit updates celsius", _set_f),
)

In [ ]:
# ✏️ Exercise 8.5 — Protect state: a read-only balance changed only through methods
#   balance      read-only property (reads self._balance)
#   deposit(n)   n must be > 0 (else ValueError); adds to the balance
#   withdraw(n)  n must be > 0 (else ValueError) and <= balance (else ValueError)
class BankAccount:
    def __init__(self, owner):
        self.owner = owner
        self._balance = 0

    pass   # <- write the balance property, deposit and withdraw

def _flow():
    a = BankAccount("Ana"); a.deposit(100); a.withdraw(30); return a.balance == 70

run_checks(
    ("new account has balance 0", lambda: BankAccount("Ana").balance == 0),
    ("deposit then withdraw", _flow),
    ("balance can't be assigned directly", lambda: raises(AttributeError, lambda: setattr(BankAccount("Ana"), "balance", 999))),
    ("deposit(-5) raises ValueError", lambda: raises(ValueError, lambda: BankAccount("Ana").deposit(-5))),
    ("withdraw(0) raises ValueError", lambda: raises(ValueError, lambda: BankAccount("Ana").withdraw(0))),
    ("overdraft raises ValueError", lambda: raises(ValueError, lambda: BankAccount("Ana").withdraw(10))),
)

---
# Part 9 · `@classmethod` & `@staticmethod`

## 📖 Learn

Normal methods receive the **object** (`self`). Two other kinds don't:

| Decorator | First parameter | Use it for |
|---|---|---|
| `@classmethod` | `cls`, the class itself | **alternative constructors** (`Point.from_string("3,4")`) |
| `@staticmethod` | nothing special | a plain helper function that logically belongs to the class |

Inside a classmethod, build the object with **`cls(...)`**, *not* the hard-coded class name. That way subclasses get objects of the right type.

## 👀 See it

In [ ]:
class Date:
    def __init__(self, year, month, day):
        self.year, self.month, self.day = year, month, day

    @classmethod
    def from_string(cls, text):                 # alternative constructor
        y, m, d = (int(part) for part in text.split("-"))
        return cls(y, m, d)

    @staticmethod
    def is_leap(year):                          # no self, no cls: just a helper
        return year % 4 == 0 and (year % 100 != 0 or year % 400 == 0)

    def __repr__(self):
        return f"{type(self).__name__}({self.year}, {self.month}, {self.day})"

d = Date.from_string("2024-11-12")
print(d, Date.is_leap(2024), d.is_leap(1900))

class Birthday(Date):
    pass

print(Birthday.from_string("2000-02-29"))        # cls made a BIRTHDAY, not a Date

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 9.1 — An alternative constructor
# Point.from_string("3,4") should return Point(3, 4) (with integers).
class Point:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    pass   # <- write the classmethod from_string(cls, text)

run_checks(
    ("from_string('3,4')", lambda: (Point.from_string("3,4").x, Point.from_string("3,4").y) == (3, 4)),
    ("negative numbers", lambda: (Point.from_string("10,-2").x, Point.from_string("10,-2").y) == (10, -2)),
    ("returns a Point", lambda: isinstance(Point.from_string("1,1"), Point)),
)

In [ ]:
# ✏️ Exercise 9.2 — Alternative constructor that does a calculation
# Person.from_birth_year("Ana", 1990, 2026) -> Person("Ana", 36)
class Person:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    pass   # <- write from_birth_year(cls, name, birth_year, current_year)

run_checks(
    ("age is computed", lambda: Person.from_birth_year("Ana", 1990, 2026).age == 36),
    ("name is passed through", lambda: Person.from_birth_year("Ana", 1990, 2026).name == "Ana"),
)

In [ ]:
# ✏️ Exercise 9.3 — Static methods: helpers that need neither self nor cls
#   TempTools.c_to_f(c)       -> c * 9/5 + 32
#   TempTools.is_freezing(c)  -> True when c <= 0
class TempTools:
    pass   # <- write both @staticmethods

run_checks(
    ("c_to_f(100) == 212", lambda: TempTools.c_to_f(100) == 212),
    ("c_to_f(0) == 32", lambda: TempTools.c_to_f(0) == 32),
    ("is_freezing(-5) is True", lambda: TempTools.is_freezing(-5) is True),
    ("is_freezing(20) is False", lambda: TempTools.is_freezing(20) is False),
    ("also callable on an object", lambda: TempTools().c_to_f(100) == 212),
)

In [ ]:
# ✏️ Exercise 9.4 — A "factory" classmethod
# Config.default() returns a NEW Config(debug=False, level=1) each time.
class Config:
    def __init__(self, debug, level):
        self.debug = debug
        self.level = level

    pass   # <- write the classmethod default(cls)

run_checks(
    ("debug is False", lambda: Config.default().debug is False),
    ("level is 1", lambda: Config.default().level == 1),
    ("each call makes a new object", lambda: Config.default() is not Config.default()),
)

In [ ]:
# ✏️ Exercise 9.5 ⭐ — Why `cls` matters
# Student.from_dict({"name": "Ana", "grade": 90}) builds a Student.
# GradStudent inherits it, so GradStudent.from_dict(...) must build a GRADSTUDENT.
class Student:
    def __init__(self, name, grade):
        self.name = name
        self.grade = grade

    pass   # <- write from_dict(cls, data). Use cls(...), NOT Student(...)

class GradStudent(Student):
    pass

run_checks(
    ("Student.from_dict works", lambda: Student.from_dict({"name": "Ana", "grade": 90}).name == "Ana"),
    ("grade is passed through", lambda: Student.from_dict({"name": "Ana", "grade": 90}).grade == 90),
    ("subclass gets a subclass object", lambda: type(GradStudent.from_dict({"name": "Bo", "grade": 80})) is GradStudent),
)

---
# Part 10 · Inheritance, `super()` & polymorphism

## 📖 Learn

**Inheritance** lets a new class start from an existing one: `class Dog(Animal):`. The child (*subclass*) gets all of the parent's attributes and methods for free, and can:

* **add** new methods,
* **override** a method by redefining it with the same name,
* **extend** the parent's version by calling **`super()`**: `super().__init__(...)`, `super().describe()`.

If a subclass defines its own `__init__`, it should usually call `super().__init__(...)` so the parent's setup still happens.

**Polymorphism:** code that calls `animal.speak()` doesn't need to know which kind of animal it has. Each class answers in its own way.

`isinstance(obj, Cls)` is `True` for the class *and all its parents*. `issubclass(Child, Parent)` checks the classes directly.

## 👀 See it

In [ ]:
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def describe(self):
        return f"{self.name} says {self.speak()}"

class Dog(Animal):
    def speak(self):                      # override
        return "Woof"

class Puppy(Dog):
    def __init__(self, name, age_months):
        super().__init__(name)            # let Animal set up name
        self.age_months = age_months

    def describe(self):                   # extend the parent's version
        return super().describe() + f" (a {self.age_months}-month-old puppy)"

for pet in [Animal("Generic"), Dog("Rex"), Puppy("Bit", 3)]:
    print(pet.describe())                 # same call, different behaviour

p = Puppy("Bit", 3)
print(isinstance(p, Puppy), isinstance(p, Dog), isinstance(p, Animal), isinstance(p, str))
print(issubclass(Puppy, Animal), [c.__name__ for c in Puppy.__mro__])

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 10.1 — Override a method
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

# <- write class Dog(Animal) whose speak() returns "Woof" (nothing else needed!)

run_checks(
    ("Dog inherits __init__", lambda: Dog("Rex").name == "Rex"),
    ("Dog speaks Woof", lambda: Dog("Rex").speak() == "Woof"),
    ("a Dog is an Animal", lambda: isinstance(Dog("Rex"), Animal)),
    ("Animal is unchanged", lambda: Animal("x").speak() == "..."),
)

In [ ]:
# ✏️ Exercise 10.2 — Polymorphism: same call, different behaviour
class Animal:
    def __init__(self, name):
        self.name = name

    def speak(self):
        return "..."

    def describe(self):
        return f"{self.name} says {self.speak()}"

# <- write Cat (speak -> "Meow") and Duck (speak -> "Quack"), both subclasses of Animal

def zoo():
    return [Cat("Tom"), Duck("Don"), Animal("Generic")]
run_checks(
    ("Cat.describe()", lambda: Cat("Tom").describe() == "Tom says Meow"),
    ("Duck.describe()", lambda: Duck("Don").describe() == "Don says Quack"),
    ("describe() works through a list of mixed animals",
     lambda: [a.describe() for a in zoo()] == ["Tom says Meow", "Don says Quack", "Generic says ..."]),
)

In [ ]:
# ✏️ Exercise 10.3 — super().__init__
# Car(brand, doors) is a Vehicle with 4 wheels. Reuse Vehicle's __init__ via super().
class Vehicle:
    def __init__(self, brand, wheels):
        self.brand = brand
        self.wheels = wheels

class Car(Vehicle):
    pass   # <- write __init__(self, brand, doors)

run_checks(
    ("brand set by the parent", lambda: Car("Fiat", 5).brand == "Fiat"),
    ("wheels is 4", lambda: Car("Fiat", 5).wheels == 4),
    ("doors stored", lambda: Car("Fiat", 5).doors == 5),
    ("a Car is a Vehicle", lambda: isinstance(Car("Fiat", 5), Vehicle)),
)

In [ ]:
# ✏️ Exercise 10.4 — Extend a parent method with super()
# Manager.describe() = Employee.describe() + " and manages N people"
class Employee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary

    def describe(self):
        return f"{self.name} earns {self.salary}"

class Manager(Employee):
    pass   # <- write __init__(self, name, salary, reports=None) and describe(self)

run_checks(
    ("describe with 2 reports", lambda: Manager("Ann", 100, ["a", "b"]).describe() == "Ann earns 100 and manages 2 people"),
    ("no reports by default", lambda: Manager("Bob", 90).describe() == "Bob earns 90 and manages 0 people"),
    ("employees are unchanged", lambda: Employee("Cy", 50).describe() == "Cy earns 50"),
    ("managers don't share report lists", lambda: (Manager("a", 1).reports, Manager("b", 1).reports)[0] is not Manager("b", 1).reports),
)

In [ ]:
# ✏️ Exercise 10.5 — A base class that must be overridden
import math

class Shape:
    def area(self):
        raise NotImplementedError("subclasses must implement area()")

# <- write Square(side) and Circle(radius): subclasses of Shape with __init__ and area()

def shapes():
    return [Square(3), Circle(1), Square(2)]
run_checks(
    ("Square(3).area() == 9", lambda: Square(3).area() == 9),
    ("Circle(1).area() is pi", lambda: abs(Circle(1).area() - math.pi) < 1e-9),
    ("both are Shapes", lambda: isinstance(Square(1), Shape) and isinstance(Circle(1), Shape)),
    ("total area through the base-class interface", lambda: abs(sum(s.area() for s in shapes()) - (9 + math.pi + 4)) < 1e-9),
    ("Shape itself refuses to compute", lambda: raises(NotImplementedError, lambda: Shape().area())),
)

In [ ]:
# ✏️ Exercise 10.6 — Predict, then run: isinstance and issubclass
class Animal:
    def __init__(self, name):
        self.name = name

class Dog(Animal):
    pass

class Cat(Animal):
    pass

d = Dog("Rex")
# Predict these four, in order:  isinstance(d, Dog)  isinstance(d, Animal)
#                                isinstance(d, Cat)  issubclass(Cat, Animal)
answers = None   # <- a list of four True/False values
run_checks(("your predictions match", lambda: answers == [isinstance(d, Dog), isinstance(d, Animal), isinstance(d, Cat), issubclass(Cat, Animal)]))

---
# Part 11 · Composition: objects that contain objects

## 📖 Learn

Inheritance models **"is-a"** (a Dog *is an* Animal). Composition models **"has-a"**: one object *holds* others in its attributes: a `Library` has `Book`s, an `Order` has `Item`s, a `Course` has `Student`s.

Most real programs are mostly composition. Prefer it whenever "has-a" describes the relationship better than "is-a".

Things to watch:

* Objects are stored **by reference**: a list of `Book` objects holds the same objects you created, not copies.
* When two objects refer to each other (a student knows its courses; a course knows its students), you must update **both sides**.

## 👀 See it

In [ ]:
class LineItem:
    def __init__(self, name, price, qty):
        self.name, self.price, self.qty = name, price, qty

class Cart:
    def __init__(self):
        self.items = []                    # a Cart HAS line items

    def add(self, item):
        self.items.append(item)

    def total(self):
        return sum(i.price * i.qty for i in self.items)

pen = LineItem("pen", 1.5, 4)
cart = Cart()
cart.add(pen)
cart.add(LineItem("book", 12.0, 1))
print(cart.total())

pen.qty = 10                               # same object, so the cart sees the change
print(cart.total())

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 11.1 — A Library that has Books
#   add(book)               store the book
#   find_by_author(author)  list of TITLES by that author (in the order added)
#   __len__                 number of books
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

# <- write class Library

def _lib():
    lib = Library()
    lib.add(Book("Dune", "Herbert"))
    lib.add(Book("Emma", "Austen"))
    lib.add(Book("Children of Dune", "Herbert"))
    return lib

run_checks(
    ("a new library is empty", lambda: len(Library()) == 0),
    ("len counts books", lambda: len(_lib()) == 3),
    ("find_by_author", lambda: _lib().find_by_author("Herbert") == ["Dune", "Children of Dune"]),
    ("unknown author gives []", lambda: _lib().find_by_author("Nobody") == []),
    ("libraries don't share books", lambda: (_lib(), len(Library()))[1] == 0),
)

In [ ]:
# ✏️ Exercise 11.2 — An Order that has Items
#   add(item)   store an Item
#   total()     sum of price * qty (0 for an empty order)
class Item:
    def __init__(self, name, price, qty):
        self.name = name
        self.price = price
        self.qty = qty

# <- write class Order

def _order():
    o = Order(); o.add(Item("pen", 3.0, 2)); o.add(Item("book", 4.5, 1)); return o

run_checks(
    ("empty order total is 0", lambda: Order().total() == 0),
    ("2*3.0 + 1*4.5 == 10.5", lambda: _order().total() == 10.5),
    ("total reflects later changes to an item", lambda: (lambda o: (setattr(o.items[0], "qty", 10), o.total())[1])(_order()) == 34.5),
)

In [ ]:
# ✏️ Exercise 11.3 — A Team that has Players
#   add_player(player)
#   best_player()     the Player with the highest score (None if the team is empty)
#   average_score()   mean score (0 if empty)
class Player:
    def __init__(self, name, score):
        self.name = name
        self.score = score

class Team:
    def __init__(self, name):
        self.name = name
        self.players = []

    pass   # <- write add_player, best_player, average_score

def _team():
    t = Team("Reds")
    for n, s in [("ann", 10), ("bob", 30), ("cy", 20)]:
        t.add_player(Player(n, s))
    return t

run_checks(
    ("best_player", lambda: _team().best_player().name == "bob"),
    ("average_score", lambda: _team().average_score() == 20),
    ("empty team: best_player is None", lambda: Team("x").best_player() is None),
    ("empty team: average is 0", lambda: Team("x").average_score() == 0),
)

In [ ]:
# ✏️ Exercise 11.4 — Two-way relationships
# course.enroll(student) must update BOTH sides:
#   the student is added to course.roster and the course is added to student.courses.
# Enrolling the same student twice must not create duplicates.
class Student:
    def __init__(self, name):
        self.name = name
        self.courses = []

class Course:
    def __init__(self, title):
        self.title = title
        self.roster = []

    pass   # <- write enroll(self, student)

def _setup():
    s, c = Student("Ana"), Course("Python")
    c.enroll(s)
    return s, c

run_checks(
    ("student is on the roster", lambda: (lambda s, c: s in c.roster)(*_setup())),
    ("course is on the student", lambda: (lambda s, c: c in s.courses)(*_setup())),
    ("no duplicates when enrolling twice", lambda: (lambda s, c: (c.enroll(s), len(c.roster), len(s.courses))[1:])(*_setup()) == (1, 1)),
)

In [ ]:
# ✏️ Exercise 11.5 ⭐ — A Deck of Cards
# __init__ builds all 52 cards (every rank in every suit); draw() removes and returns the LAST card;
# __len__ gives how many cards are left. Drawing from an empty deck should raise IndexError.
class Card:
    def __init__(self, rank, suit):
        self.rank = rank
        self.suit = suit

    def __repr__(self):
        return f"{self.rank}{self.suit}"

RANKS = ["A", "2", "3", "4", "5", "6", "7", "8", "9", "10", "J", "Q", "K"]
SUITS = ["S", "H", "D", "C"]

# <- write class Deck

def _drain():
    d = Deck()
    for _ in range(52):
        d.draw()
    return len(d) == 0 and raises(IndexError, d.draw)

run_checks(
    ("a deck has 52 cards", lambda: len(Deck()) == 52),
    ("all 52 cards are distinct", lambda: len({(c.rank, c.suit) for c in Deck().cards}) == 52),
    ("draw returns a Card", lambda: isinstance(Deck().draw(), Card)),
    ("draw removes the card", lambda: (lambda d: (d.draw(), len(d))[1])(Deck()) == 51),
    ("drawing everything, then one more, raises IndexError", _drain),
    ("decks don't share cards", lambda: (lambda a, b: (a.draw(), len(b))[1])(Deck(), Deck()) == 52),
)

---
# Part 12 · Container protocols & `@dataclass`

## 📖 Learn

**Container protocols.** Make your object behave like a collection by defining:

| Method | Enables |
|---|---|
| `__len__` | `len(obj)` |
| `__getitem__(i)` | `obj[i]`, and also iteration with `for` |
| `__iter__` | `for x in obj` (a generator with `yield` is the easiest way) |
| `__contains__(x)` | `x in obj` |

**Dataclasses.** Most "data holder" classes are the same boilerplate: `__init__`, `__repr__`, `__eq__`. The `@dataclass` decorator writes them for you from *annotated fields*:

```python
from dataclasses import dataclass, field

@dataclass
class Product:
    name: str
    price: float
    quantity: int = 0                         # a default
    tags: list = field(default_factory=list)  # mutable defaults need default_factory!
```

Useful options: `@dataclass(order=True)` (adds `<`, `>`, sorting by fields in order), `@dataclass(frozen=True)` (immutable and hashable). Run extra setup in **`__post_init__`**, and use `field(init=False)` for values you compute rather than receive.

## 👀 See it

In [ ]:
class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):                      # a generator: each yield produces one item
        n = self.start
        while n > 0:
            yield n
            n -= 1

    def __contains__(self, x):
        return 0 < x <= self.start

print(list(Countdown(3)), 2 in Countdown(3), 9 in Countdown(3))
print(sum(Countdown(4)))

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Gadget:
    name: str
    price: float
    quantity: int = 0
    tags: list = field(default_factory=list)

    def total_value(self):                   # you can still add methods
        return self.price * self.quantity

g = Gadget("pen", 1.5, 10)
print(g)                                     # free __repr__
print(g == Gadget("pen", 1.5, 10))           # free __eq__
print(g.total_value())

@dataclass(frozen=True, order=True)
class Release:
    major: int
    minor: int

print(sorted([Release(1, 10), Release(1, 2), Release(0, 9)]))
try:
    Release(1, 0).major = 2
except Exception as e:
    print(type(e).__name__)

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 12.1 — __getitem__ makes obj[i] AND for-loops work
class Playlist:
    def __init__(self, songs):
        self.songs = songs

    def __len__(self):
        return len(self.songs)

    pass   # <- write __getitem__(self, index)

p = Playlist(["a", "b", "c"])
run_checks(
    ("p[0]", lambda: p[0] == "a"),
    ("p[-1]", lambda: p[-1] == "c"),
    ("for-loops work", lambda: [s for s in p] == ["a", "b", "c"]),
    ("list(p) works", lambda: list(p) == ["a", "b", "c"]),
    ("out of range raises IndexError", lambda: raises(IndexError, lambda: p[10])),
)

In [ ]:
# ✏️ Exercise 12.2 — __iter__ with yield
# Countdown(3) should iterate 3, 2, 1. It must be usable more than once.
class Countdown:
    def __init__(self, start):
        self.start = start

    pass   # <- write __iter__(self) as a generator

c = Countdown(3)
run_checks(
    ("list(Countdown(3)) == [3, 2, 1]", lambda: list(c) == [3, 2, 1]),
    ("can be iterated a second time", lambda: list(c) == [3, 2, 1]),
    ("Countdown(0) is empty", lambda: list(Countdown(0)) == []),
    ("works with sum()", lambda: sum(Countdown(4)) == 10),
)

In [ ]:
# ✏️ Exercise 12.3 — __contains__ (the `in` operator)
# A Roster is case-INsensitive: "ANN" in roster is True if "ann" is on it.
class Roster:
    def __init__(self, names):
        self.names = names

    pass   # <- write __contains__(self, name)

r = Roster(["ann", "Bob"])
run_checks(
    ("'ann' in roster", lambda: "ann" in r),
    ("case-insensitive", lambda: "ANN" in r and "bob" in r),
    ("'cy' not in roster", lambda: "cy" not in r),
)

In [ ]:
# ✏️ Exercise 12.4 — Your first @dataclass
# Product has: name (str), price (float), quantity (int, default 0)
from dataclasses import dataclass, field

# <- write the dataclass

run_checks(
    ("fields stored", lambda: (Product("pen", 1.5, 3).name, Product("pen", 1.5, 3).price, Product("pen", 1.5, 3).quantity) == ("pen", 1.5, 3)),
    ("quantity defaults to 0", lambda: Product("pen", 1.5).quantity == 0),
    ("equal by value", lambda: Product("pen", 1.5) == Product("pen", 1.5, 0)),
    ("repr shows the fields", lambda: repr(Product("pen", 1.5)).startswith("Product(name='pen'")),
)

In [ ]:
# ✏️ Exercise 12.5 — Mutable defaults in a dataclass need default_factory
# Basket has one field: items, a list that defaults to a NEW empty list for each Basket.
from dataclasses import dataclass, field

# <- write the dataclass

def _separate():
    a, b = Basket(), Basket()
    a.items.append("x")
    return b.items == []

run_checks(
    ("items defaults to []", lambda: Basket().items == []),
    ("baskets don't share the list", _separate),
    ("you can pass items", lambda: Basket(["a"]).items == ["a"]),
)

In [ ]:
# ✏️ Exercise 12.6 — Frozen + ordered dataclass
# Version(major, minor): sortable by (major, minor), immutable, and usable in sets/dict keys.
from dataclasses import dataclass

# <- write the dataclass with the right options

run_checks(
    ("sorting", lambda: sorted([Version(1, 10), Version(1, 2), Version(0, 9)]) == [Version(0, 9), Version(1, 2), Version(1, 10)]),
    ("comparison", lambda: Version(1, 2) < Version(1, 10)),
    ("immutable", lambda: raises(AttributeError, lambda: setattr(Version(1, 0), "major", 2))),
    ("hashable: equal versions collapse in a set", lambda: len({Version(1, 0), Version(1, 0), Version(2, 0)}) == 2),
)

In [ ]:
# ✏️ Exercise 12.7 ⭐ — __post_init__: validate and compute
# Rectangle(width, height): both must be > 0 (else ValueError).
# It also has an `area` field that is NOT an __init__ argument; it is computed from width*height.
from dataclasses import dataclass, field

# <- write the dataclass (hint: area: float = field(init=False))

run_checks(
    ("area is computed", lambda: Rectangle(3, 4).area == 12),
    ("width and height stored", lambda: (Rectangle(3, 4).width, Rectangle(3, 4).height) == (3, 4)),
    ("negative width raises ValueError", lambda: raises(ValueError, lambda: Rectangle(-1, 4))),
    ("zero height raises ValueError", lambda: raises(ValueError, lambda: Rectangle(3, 0))),
)

---
# Part 13 · Mini-projects

Now combine everything: `__init__`, methods, validation, dunder methods, custom exceptions. Each spec is in the comments. Read the checks to see exactly what's expected.

> **Custom exceptions** are just empty subclasses of `Exception`:
> ```python
> class InsufficientFundsError(Exception):
>     pass
> ```
> Raise them like any other: `raise InsufficientFundsError("need more money")`.

In [ ]:
# ✏️ Exercise 13.1 — BankAccount with a custom exception and a history
#   class InsufficientFundsError(Exception)
#   BankAccount(owner, balance=0)
#     .deposit(amount)   amount must be > 0 (ValueError); adds to balance; records ("deposit", amount)
#     .withdraw(amount)  amount must be > 0 (ValueError); if amount > balance raise
#                        InsufficientFundsError; subtracts; records ("withdraw", amount)
#     .history           a list of (kind, amount) tuples. Failed operations are NOT recorded.
# write your exception and class here

def _flow():
    a = BankAccount("Ana", 50)
    a.deposit(100); a.withdraw(30)
    return a.balance == 120 and a.history == [("deposit", 100), ("withdraw", 30)]

def _overdraft_safe():
    a = BankAccount("Ana", 10)
    try:
        a.withdraw(50)
    except InsufficientFundsError:
        pass
    return a.balance == 10 and a.history == []

run_checks(
    ("deposit and withdraw update balance and history", _flow),
    ("overdraft raises InsufficientFundsError", lambda: raises(InsufficientFundsError, lambda: BankAccount("A", 10).withdraw(50))),
    ("a failed withdraw changes nothing", _overdraft_safe),
    ("InsufficientFundsError is an Exception", lambda: issubclass(InsufficientFundsError, Exception)),
    ("deposit(0) raises ValueError", lambda: raises(ValueError, lambda: BankAccount("A").deposit(0))),
    ("withdraw(-1) raises ValueError", lambda: raises(ValueError, lambda: BankAccount("A", 5).withdraw(-1))),
    ("default balance is 0 and history is separate per account", lambda: (lambda a, b: (a.deposit(5), b.history)[1])(BankAccount("a"), BankAccount("b")) == []),
)

In [ ]:
# ✏️ Exercise 13.2 — Inventory (a dict inside an object + 3 dunder methods)
#   add_stock(item, qty)     qty must be > 0 (ValueError); adds to what's there
#   remove_stock(item, qty)  ValueError if the item is unknown or qty exceeds stock
#   low_stock(threshold)     SORTED list of item names with quantity < threshold
#   __len__                  number of distinct items
#   __contains__(item)       is this item known?
# write your class here

def _inv():
    inv = Inventory()
    inv.add_stock("pen", 10); inv.add_stock("cap", 2); inv.add_stock("ink", 1)
    return inv

run_checks(
    ("len counts distinct items", lambda: len(_inv()) == 3),
    ("'in' works", lambda: "pen" in _inv() and "tape" not in _inv()),
    ("adding twice accumulates", lambda: (lambda i: (i.add_stock("pen", 5), i.low_stock(100))[1])(_inv()) == ["cap", "ink", "pen"]),
    ("low_stock(5) is sorted", lambda: _inv().low_stock(5) == ["cap", "ink"]),
    ("remove_stock reduces", lambda: (lambda i: (i.remove_stock("pen", 8), i.low_stock(3))[1])(_inv()) == ["cap", "ink", "pen"]),
    ("can't remove more than we have", lambda: raises(ValueError, lambda: _inv().remove_stock("cap", 5))),
    ("can't remove an unknown item", lambda: raises(ValueError, lambda: _inv().remove_stock("tape", 1))),
    ("add_stock(0) raises ValueError", lambda: raises(ValueError, lambda: _inv().add_stock("pen", 0))),
)

In [ ]:
# ✏️ Exercise 13.3 — A TodoList that holds Task objects
#   add(title)        create a Task and store it
#   complete(title)   mark the task done; raise KeyError if there's no such task
#   pending()         titles of tasks NOT done, in the order added
#   __len__           total number of tasks
from dataclasses import dataclass

@dataclass
class Task:
    title: str
    done: bool = False

# <- write class TodoList

def _todo():
    t = TodoList()
    for title in ["write", "test", "ship"]:
        t.add(title)
    return t

run_checks(
    ("len counts tasks", lambda: len(_todo()) == 3),
    ("everything is pending at first", lambda: _todo().pending() == ["write", "test", "ship"]),
    ("complete removes a task from pending()", lambda: (lambda t: (t.complete("test"), t.pending())[1])(_todo()) == ["write", "ship"]),
    ("completed tasks still count in len()", lambda: (lambda t: (t.complete("test"), len(t))[1])(_todo()) == 3),
    ("unknown task raises KeyError", lambda: raises(KeyError, lambda: _todo().complete("nope"))),
)

In [ ]:
# ✏️ Exercise 13.4 — A Stack (last in, first out)
#   push(x), pop() -> the most recently pushed item (IndexError if empty),
#   peek() -> look at the top without removing (IndexError if empty),
#   is_empty(), and __len__
# write your class here

def _stack():
    s = Stack()
    s.push(1); s.push(2); s.push(3)
    return s

run_checks(
    ("starts empty", lambda: Stack().is_empty() and len(Stack()) == 0),
    ("pop returns the last pushed", lambda: _stack().pop() == 3),
    ("pop removes it", lambda: (lambda s: (s.pop(), len(s), s.peek())[1:])(_stack()) == (2, 2)),
    ("peek doesn't remove", lambda: (lambda s: (s.peek(), len(s)))(_stack()) == (3, 3)),
    ("pop on empty raises IndexError", lambda: raises(IndexError, lambda: Stack().pop())),
    ("peek on empty raises IndexError", lambda: raises(IndexError, lambda: Stack().peek())),
    ("stacks are independent", lambda: (lambda a, b: (a.push(1), len(b))[1])(Stack(), Stack()) == 0),
)

In [ ]:
# ✏️ Exercise 13.5 ⭐ — A Frac (fraction) class
#   Frac(num, den=1)   den == 0 -> ValueError. Always store in lowest terms with a POSITIVE
#                      denominator:  Frac(2, 4) -> 1/2,  Frac(1, -2) -> -1/2
#   attributes         num, den
#   +  and  *          return a NEW Frac
#   ==                 equal by value (False/NotImplemented for non-Fracs)
#   str(f)  -> "5/6"   repr(f) -> "Frac(5, 6)"
# Hint: math.gcd
import math

# write your class here

run_checks(
    ("reduces to lowest terms", lambda: (Frac(2, 4).num, Frac(2, 4).den) == (1, 2)),
    ("sign lives in the numerator", lambda: (Frac(1, -2).num, Frac(1, -2).den) == (-1, 2)),
    ("zero denominator raises ValueError", lambda: raises(ValueError, lambda: Frac(1, 0))),
    ("1/2 + 1/3 == 5/6", lambda: Frac(1, 2) + Frac(1, 3) == Frac(5, 6)),
    ("2/3 * 3/4 == 1/2", lambda: Frac(2, 3) * Frac(3, 4) == Frac(1, 2)),
    ("equality is by value", lambda: Frac(1, 2) == Frac(2, 4) and Frac(1, 2) != Frac(1, 3)),
    ("comparing with an int is False", lambda: (Frac(1, 1) == 1) is False),
    ("str", lambda: str(Frac(5, 6)) == "5/6"),
    ("repr", lambda: repr(Frac(5, 6)) == "Frac(5, 6)"),
    ("Frac(3) is 3/1", lambda: (Frac(3).num, Frac(3).den) == (3, 1)),
)

In [ ]:
# ✏️ Exercise 13.6 ⭐ — An EventEmitter: objects can hold FUNCTIONS too
#   on(event, handler)   register a function for an event name
#   emit(event, *args)   call every handler registered for that event with *args,
#                        in registration order; return how many handlers were called
#                        (0 for an event nobody registered; that's not an error)
# write your class here

def _two_handlers():
    seen = []
    e = EventEmitter()
    e.on("greet", lambda name: seen.append(f"hi {name}"))
    e.on("greet", lambda name: seen.append(f"yo {name}"))
    n = e.emit("greet", "ana")
    return n == 2 and seen == ["hi ana", "yo ana"]

def _separate_events():
    seen = []
    e = EventEmitter()
    e.on("a", lambda: seen.append("a"))
    e.on("b", lambda: seen.append("b"))
    e.emit("b")
    return seen == ["b"]

run_checks(
    ("two handlers, called in order, with the arguments", _two_handlers),
    ("only the matching event's handlers run", _separate_events),
    ("an event with no handlers returns 0", lambda: EventEmitter().emit("nothing") == 0),
    ("emitters don't share handlers", lambda: (lambda a, b: (a.on("x", lambda: None), b.emit("x"))[1])(EventEmitter(), EventEmitter()) == 0),
)

---
# Part 14 · Classes + pandas

## 📖 Learn

You've been using classes all along: `pd.DataFrame` **is a class**. `df.shape` is an attribute, `df.head()` is a method, `len(df)` calls `DataFrame.__len__`. So everything you just learned applies in two directions:

* **Your objects → pandas:** a list of dataclass objects becomes a DataFrame with `pd.DataFrame([asdict(o) for o in objs])`.
* **pandas → your objects:** `df.to_dict("records")` gives a list of dicts you can feed to your class: `Person(**row)`.
* **Wrap a DataFrame in your own class** to give it a clean, meaningful interface (`report.by_region()` instead of repeating `groupby` code everywhere).

## 👀 See it

In [ ]:
import pandas as pd
from dataclasses import dataclass, asdict

@dataclass
class Person:
    name: str
    age: int

people = [Person("Ann", 30), Person("Bob", 25)]
df = pd.DataFrame([asdict(p) for p in people])      # objects -> DataFrame
print(df)

back = [Person(**row) for row in df.to_dict("records")]   # DataFrame -> objects
print(back, back == people)

print(type(df), df.shape, len(df))                  # a DataFrame is just an object

In [ ]:
class SalesData:
    """A thin wrapper that gives a DataFrame a friendlier interface."""
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def total(self):
        return self.df["amount"].sum()

sales = SalesData(pd.DataFrame({"region": ["N", "S", "N"], "amount": [100, 50, 25]}))
print(len(sales), sales.total())

## ✏️ Your turn

In [ ]:
# ✏️ Exercise 14.1 — Objects -> DataFrame
import pandas as pd
from dataclasses import dataclass, asdict

@dataclass
class Person:
    name: str
    age: int

people = [Person("Ann", 30), Person("Bob", 25), Person("Cy", 41)]
df = None   # <- build a DataFrame with columns name and age from the list of objects

run_checks(
    ("it's a DataFrame", lambda: isinstance(df, pd.DataFrame)),
    ("columns are name and age", lambda: list(df.columns) == ["name", "age"]),
    ("ages", lambda: df["age"].tolist() == [30, 25, 41]),
    ("names", lambda: df["name"].tolist() == ["Ann", "Bob", "Cy"]),
)

In [ ]:
# ✏️ Exercise 14.2 — Wrap a DataFrame in a class
#   SalesReport(df)
#     .total()       the sum of the "amount" column
#     .by_region()   a dict {region: total amount}
import pandas as pd

sales_df = pd.DataFrame({
    "region": ["N", "S", "N", "E"],
    "amount": [100, 50, 25, 10],
})

# <- write class SalesReport

run_checks(
    ("total() == 185", lambda: SalesReport(sales_df).total() == 185),
    ("by_region()", lambda: SalesReport(sales_df).by_region() == {"N": 125, "S": 50, "E": 10}),
    ("a report keeps its own data", lambda: SalesReport(sales_df.head(1)).total() == 100),
)

In [ ]:
# ✏️ Exercise 14.3 — Methods that return NEW objects (don't mutate)
#   drop_missing()  returns a NEW Dataset with rows containing NaN removed; the original is unchanged
#   columns         a property: the list of column names
import pandas as pd

class Dataset:
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    pass   # <- write drop_missing(self) and the columns property

raw = pd.DataFrame({"a": [1, None, 3], "b": ["x", "y", None]})
ds = Dataset(raw)
run_checks(
    ("drop_missing returns a Dataset", lambda: isinstance(ds.drop_missing(), Dataset)),
    ("only the complete row is left", lambda: len(ds.drop_missing()) == 1),
    ("the original is untouched", lambda: len(ds) == 3),
    ("it's a different object", lambda: ds.drop_missing() is not ds),
    ("columns property", lambda: ds.columns == ["a", "b"]),
)

In [ ]:
# ✏️ Exercise 14.4 — DataFrame -> list of objects
# Write rows_to_people(df) returning a list of Person objects, one per row.
import pandas as pd
from dataclasses import dataclass

@dataclass
class Person:
    name: str
    age: int

# <- write the function rows_to_people(df)

df = pd.DataFrame({"name": ["Ann", "Bob"], "age": [30, 25]})
run_checks(
    ("returns a list of Person", lambda: all(isinstance(p, Person) for p in rows_to_people(df))),
    ("values match", lambda: rows_to_people(df) == [Person("Ann", 30), Person("Bob", 25)]),
    ("empty frame gives []", lambda: rows_to_people(df.iloc[0:0]) == []),
)

---
# 🎓 Wrap-up

## Cheat sheet

```python
class Name(Parent):                       # (Parent) is optional
    class_attr = 0                        # shared by all objects

    def __init__(self, a, b=None):        # runs on Name(...)
        super().__init__()                # only if you have a parent
        self.a = a                        # instance attributes
        self.b = b if b is not None else []

    def method(self):                     # self = the object
        return self.a

    @property
    def computed(self):                   # obj.computed (no parentheses)
        return self.a * 2

    @computed.setter
    def computed(self, v):                # obj.computed = 5
        self.a = v / 2

    @classmethod
    def from_x(cls, x):                   # alternative constructor
        return cls(x)

    @staticmethod
    def helper(n):                        # plain function, namespaced
        return n + 1

    def __repr__(self): ...               # developer text    (print() falls back to it)
    def __str__(self): ...                # user text
    def __eq__(self, other): ...          # ==
    def __lt__(self, other): ...          # <  (enables sorted)
    def __add__(self, other): ...         # +
    def __len__(self): ...                # len()
    def __iter__(self): ...               # for x in obj
    def __contains__(self, x): ...        # x in obj
```

## Common mistakes (and what Python tells you)

| Symptom | Usual cause |
|---|---|
| `TypeError: __init__() takes 1 positional argument but 2 were given` | you forgot `self` as the first parameter |
| `NameError: name 'name' is not defined` inside a method | you wrote `name` instead of `self.name` |
| `AttributeError: 'Dog' object has no attribute 'name'` | `__init__` never set it, or misspelled |
| attribute is set but the next object has it too | mutable default (`=[]`) or a class attribute you meant to be per-object |
| `print(obj)` shows `<__main__.Dog object at 0x...>` | add `__str__` / `__repr__` |
| `__str__` shows `None` | you used `print` instead of `return` inside it |
| `==` is False for "identical" objects | define `__eq__` (or use `@dataclass`) |
| `TypeError: '<' not supported between instances` | define `__lt__` (or `@dataclass(order=True)`) |
| subclass loses parent's attributes | its `__init__` didn't call `super().__init__(...)` |

## Habits that save hours

* **Name classes with `CamelCase`, methods and attributes with `snake_case`.**
* Make objects **valid from the moment they're created**: validate in `__init__`.
* Give every class a useful **`__repr__`** (or use `@dataclass`). Debugging gets much easier.
* **Prefer composition** ("has-a") over inheritance ("is-a") unless the "is-a" is truly natural.
* Use `isinstance(...)` instead of comparing `type(x) == ...`.
* If a method doesn't use `self`, ask whether it should be a `@staticmethod`, or just a function.

## Where next?

* Redo the exercises you got wrong *without* the answer key, tomorrow.
* Rewrite one of your own earlier scripts as a class (a bank, a gradebook, a game character...).
* Explore `abc.ABC` and `@abstractmethod` (enforced abstract base classes), `__slots__`, `typing`, and `functools.total_ordering`.
* Look at the source of a small library class (for example `collections.namedtuple` or `dataclasses`) and see which of these ideas you recognise.

---
# 🔑 Answer key

Try every exercise first. Your solution can differ from mine and still be correct, as long as the checker is happy.

<details>
<summary><b>🔑 Answer key — click to open (try first!)</b></summary>

<pre>
1.1    text.upper()
1.2    text.split()
1.3    nums.sort()
1.4    type(3.5)
1.5    z.imag
2.1    class Cat:
           pass
2.2    b1.title = "Dune"
       b2.title = "Emma"
2.3    my_book = Book()
       my_book.title = "Dune"
       my_book.author = "Frank Herbert"
       my_book.pages = 412
2.4    "Emma"
2.5    False
3.1    def __init__(self, x, y):
           self.x = x
           self.y = y
3.2    def __init__(self, title, author, pages):
           self.title = title
           self.author = author
           self.pages = pages
3.3    def __init__(self, width, height):
           self.width = width
           self.height = height
3.4    def __init__(self, first, last):
           self.first = first
           self.last = last
           self.full_name = first + " " + last
3.5    def __init__(self):
           self.count = 0
3.6    def __init__(self, name):
           self.name = name
           self.songs = []
4.1    def greet(self):
           return f"Hello, {self.name}!"
4.2    def area(self):
           return self.width * self.height
       
       def perimeter(self):
           return 2 * (self.width + self.height)
4.3    def increment(self):
           self.count += 1
       
       def reset(self):
           self.count = 0
4.4    def area(self):
           return math.pi * self.radius ** 2
       
       def circumference(self):
           return 2 * math.pi * self.radius
4.5    def add_grade(self, grade):
           self.grades.append(grade)
       
       def average(self):
           if not self.grades:
               return 0
           return sum(self.grades) / len(self.grades)
4.6    def describe(self):
           return f"{self.width}x{self.height} rectangle with area {self.area()}"
5.1    def __init__(self, name, price=0.0, quantity=1):
           self.name = name
           self.price = price
           self.quantity = quantity
5.2    def __init__(self, owner, balance=0):
           if balance &lt; 0:
               raise ValueError("balance cannot be negative")
           self.owner = owner
           self.balance = balance
5.3    def __init__(self, value):
           if not 0 &lt;= value &lt;= 100:
               raise ValueError("value must be between 0 and 100")
           self.value = value
5.4    def __init__(self, items=None):
           self.items = items if items is not None else []
6.1    species = "Canis familiaris"
6.2    count = 0
       
       def __init__(self, name):
           self.name = name
           Player.count += 1
6.3    (1, 2, 1)
6.4    raise_pct = 0.05
       
       def __init__(self, name, salary):
           self.name = name
           self.salary = salary
       
       def apply_raise(self):
           self.salary = self.salary * (1 + self.raise_pct)
6.5    next_id = 1
       
       def __init__(self):
           self.id = Ticket.next_id
           Ticket.next_id += 1
7.1    def __str__(self):
           return f"{self.name} (age {self.age})"
7.2    def __repr__(self):
           return f"Point({self.x}, {self.y})"
7.3    def __eq__(self, other):
           if not isinstance(other, Point):
               return False
           return self.x == other.x and self.y == other.y
7.4    def __add__(self, other):
           return Vector(self.x + other.x, self.y + other.y)
7.5    def __lt__(self, other):
           return self.grade &lt; other.grade
7.6    def __len__(self):
           return len(self.songs)
7.7    def __mul__(self, k):
           return Vector(self.x * k, self.y * k)
       
       def __neg__(self):
           return Vector(-self.x, -self.y)
8.1    @property
       def area(self):
           return self.width * self.height
8.2    @property
       def full_name(self):
           return f"{self.first} {self.last}"
8.3    @property
       def age(self):
           return self._age
       
       @age.setter
       def age(self, value):
           if value &lt; 0:
               raise ValueError("age cannot be negative")
           self._age = value
8.4    @property
       def fahrenheit(self):
           return self.celsius * 9 / 5 + 32
       
       @fahrenheit.setter
       def fahrenheit(self, value):
           self.celsius = (value - 32) * 5 / 9
8.5    @property
       def balance(self):
           return self._balance
       
       def deposit(self, amount):
           if amount &lt;= 0:
               raise ValueError("amount must be positive")
           self._balance += amount
       
       def withdraw(self, amount):
           if amount &lt;= 0:
               raise ValueError("amount must be positive")
           if amount &gt; self._balance:
               raise ValueError("insufficient funds")
           self._balance -= amount
9.1    @classmethod
       def from_string(cls, text):
           x, y = text.split(",")
           return cls(int(x), int(y))
9.2    @classmethod
       def from_birth_year(cls, name, birth_year, current_year):
           return cls(name, current_year - birth_year)
9.3    @staticmethod
       def c_to_f(c):
           return c * 9 / 5 + 32
       
       @staticmethod
       def is_freezing(c):
           return c &lt;= 0
9.4    @classmethod
       def default(cls):
           return cls(False, 1)
9.5    @classmethod
       def from_dict(cls, data):
           return cls(data["name"], data["grade"])
10.1   class Dog(Animal):
           def speak(self):
               return "Woof"
10.2   class Cat(Animal):
           def speak(self):
               return "Meow"
       
       class Duck(Animal):
           def speak(self):
               return "Quack"
10.3   def __init__(self, brand, doors):
           super().__init__(brand, 4)
           self.doors = doors
10.4   def __init__(self, name, salary, reports=None):
           super().__init__(name, salary)
           self.reports = reports if reports is not None else []
       
       def describe(self):
           return super().describe() + f" and manages {len(self.reports)} people"
10.5   class Square(Shape):
           def __init__(self, side):
               self.side = side
       
           def area(self):
               return self.side ** 2
       
       class Circle(Shape):
           def __init__(self, radius):
               self.radius = radius
       
           def area(self):
               return math.pi * self.radius ** 2
10.6   [True, True, False, True]
11.1   class Library:
           def __init__(self):
               self.books = []
       
           def add(self, book):
               self.books.append(book)
       
           def find_by_author(self, author):
               return [b.title for b in self.books if b.author == author]
       
           def __len__(self):
               return len(self.books)
11.2   class Order:
           def __init__(self):
               self.items = []
       
           def add(self, item):
               self.items.append(item)
       
           def total(self):
               return sum(i.price * i.qty for i in self.items)
11.3   def add_player(self, player):
           self.players.append(player)
       
       def best_player(self):
           if not self.players:
               return None
           return max(self.players, key=lambda p: p.score)
       
       def average_score(self):
           if not self.players:
               return 0
           return sum(p.score for p in self.players) / len(self.players)
11.4   def enroll(self, student):
           if student not in self.roster:
               self.roster.append(student)
           if self not in student.courses:
               student.courses.append(self)
11.5   class Deck:
           def __init__(self):
               self.cards = [Card(r, s) for s in SUITS for r in RANKS]
       
           def draw(self):
               return self.cards.pop()
       
           def __len__(self):
               return len(self.cards)
12.1   def __getitem__(self, index):
           return self.songs[index]
12.2   def __iter__(self):
           n = self.start
           while n &gt; 0:
               yield n
               n -= 1
12.3   def __contains__(self, name):
           return name.lower() in [n.lower() for n in self.names]
12.4   @dataclass
       class Product:
           name: str
           price: float
           quantity: int = 0
12.5   @dataclass
       class Basket:
           items: list = field(default_factory=list)
12.6   @dataclass(frozen=True, order=True)
       class Version:
           major: int
           minor: int
12.7   @dataclass
       class Rectangle:
           width: float
           height: float
           area: float = field(init=False)
       
           def __post_init__(self):
               if self.width &lt;= 0 or self.height &lt;= 0:
                   raise ValueError("width and height must be positive")
               self.area = self.width * self.height
13.1   class InsufficientFundsError(Exception):
           pass
       
       class BankAccount:
           def __init__(self, owner, balance=0):
               self.owner = owner
               self.balance = balance
               self.history = []
       
           def deposit(self, amount):
               if amount &lt;= 0:
                   raise ValueError("amount must be positive")
               self.balance += amount
               self.history.append(("deposit", amount))
       
           def withdraw(self, amount):
               if amount &lt;= 0:
                   raise ValueError("amount must be positive")
               if amount &gt; self.balance:
                   raise InsufficientFundsError(f"balance is only {self.balance}")
               self.balance -= amount
               self.history.append(("withdraw", amount))
13.2   class Inventory:
           def __init__(self):
               self.stock = {}
       
           def add_stock(self, item, qty):
               if qty &lt;= 0:
                   raise ValueError("qty must be positive")
               self.stock[item] = self.stock.get(item, 0) + qty
       
           def remove_stock(self, item, qty):
               if item not in self.stock or qty &gt; self.stock[item]:
                   raise ValueError("not enough stock")
               self.stock[item] -= qty
       
           def low_stock(self, threshold):
               return sorted(name for name, q in self.stock.items() if q &lt; threshold)
       
           def __len__(self):
               return len(self.stock)
       
           def __contains__(self, item):
               return item in self.stock
13.3   class TodoList:
           def __init__(self):
               self.tasks = []
       
           def add(self, title):
               self.tasks.append(Task(title))
       
           def complete(self, title):
               for task in self.tasks:
                   if task.title == title:
                       task.done = True
                       return
               raise KeyError(title)
       
           def pending(self):
               return [t.title for t in self.tasks if not t.done]
       
           def __len__(self):
               return len(self.tasks)
13.4   class Stack:
           def __init__(self):
               self._items = []
       
           def push(self, x):
               self._items.append(x)
       
           def pop(self):
               if not self._items:
                   raise IndexError("pop from empty stack")
               return self._items.pop()
       
           def peek(self):
               if not self._items:
                   raise IndexError("peek at empty stack")
               return self._items[-1]
       
           def is_empty(self):
               return len(self._items) == 0
       
           def __len__(self):
               return len(self._items)
13.5   class Frac:
           def __init__(self, num, den=1):
               if den == 0:
                   raise ValueError("denominator cannot be zero")
               g = math.gcd(num, den)
               if den &lt; 0:
                   g = -g
               self.num = num // g
               self.den = den // g
       
           def __add__(self, other):
               return Frac(self.num * other.den + other.num * self.den, self.den * other.den)
       
           def __mul__(self, other):
               return Frac(self.num * other.num, self.den * other.den)
       
           def __eq__(self, other):
               if not isinstance(other, Frac):
                   return False
               return (self.num, self.den) == (other.num, other.den)
       
           def __str__(self):
               return f"{self.num}/{self.den}"
       
           def __repr__(self):
               return f"Frac({self.num}, {self.den})"
13.6   class EventEmitter:
           def __init__(self):
               self._handlers = {}
       
           def on(self, event, handler):
               self._handlers.setdefault(event, []).append(handler)
       
           def emit(self, event, *args):
               handlers = self._handlers.get(event, [])
               for handler in handlers:
                   handler(*args)
               return len(handlers)
14.1   pd.DataFrame([asdict(p) for p in people])
14.2   class SalesReport:
           def __init__(self, df):
               self.df = df
       
           def total(self):
               return self.df["amount"].sum()
       
           def by_region(self):
               return self.df.groupby("region")["amount"].sum().to_dict()
14.3   def drop_missing(self):
           return Dataset(self.df.dropna())
       
       @property
       def columns(self):
           return list(self.df.columns)
14.4   def rows_to_people(df):
           return [Person(**row) for row in df.to_dict("records")]
</pre>

</details>